# Isoglot 05: Repair B training data

Notebook 03 showed Tiny Aya does not recognise many harmful requests in Hausa, Yoruba and Igbo.
This builds data that teaches it to tell harmful from benign in those languages:

- **harmful** seeds (BeaverTails 30k_train) -> a short, clear refusal in each language
- **benign** seeds that look risky (OR-Bench hard-1k) and plain ones (Alpaca) -> a helpful answer

Claude Sonnet 5.5 writes each request and reply in Hausa, Yoruba, Igbo and English; the seed's
source decides refuse vs help. Seeds near any TukaBench goal are dropped. **CuratorKIT** then
cleans, deduplicates (MinHash), runs its secrets gate and exports chat-format train/validation
splits that SafeTune reads directly (notebook 06).

**Before running:** Accelerator **None**, Internet **on**; Secrets **`HF_TOKEN`** and
**`ANTHROPIC_API_KEY`** ticked. **Save Version -> Save & Run All (Commit)**.
The pilot stops the run before spending if the projected cost is over `CAP`.


In [ ]:
%%capture
!pip install -q "isoglot[judge,mechanism,data] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os, json, collections
from pathlib import Path
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = s.get_secret("ANTHROPIC_API_KEY")
from huggingface_hub import HfApi, whoami
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
OUT = Path("/kaggle/working/traindata")


## Seeds, with test-set overlap removed

In [ ]:
from isoglot.data import load_tukabench
from isoglot.traindata import decontaminate, load_seeds

seeds = load_seeds(n_harmful=120, n_edgy=90, n_plain=30, seed=0)
goals = sorted({x.metadata["goal_en"] for x in load_tukabench()})
seeds, dropped = decontaminate(seeds, goals, threshold=0.5)
print(len(seeds), "seeds kept |", len(dropped), "dropped as close to a TukaBench goal")
print(collections.Counter(x["source"] for x in seeds))
for d in dropped[:5]:
    print("  dropped:", d["prompt"][:80], "| jaccard", d["max_eval_jaccard"])


## Pilot: 4 seeds, then project the cost

In [ ]:
from isoglot.claude import CLAUDE_PRICES, SpendMeter, client_from_env
from isoglot.traindata import generate_rows

MODEL = "claude-sonnet-5-5"
CAP = 6.00    # USD, hard stop for this notebook
meter = SpendMeter(CAP, *CLAUDE_PRICES[MODEL])
client = client_from_env()

pilot = [x for x in seeds if x["kind"] == "harmful"][:2] + [x for x in seeds if x["source"] == "or-bench-hard"][:2]
rows, skipped = generate_rows(client, MODEL, meter, pilot)
for r in rows:
    print(f"[{r['kind']}/{r['lang']}] {r['instruction'][:90]}")
    print(f"    -> {r['output'][:160]}")
print("skipped:", skipped)
per_seed = meter.spent_usd / len(pilot)
projected = per_seed * len(seeds)
print(f"pilot ${meter.spent_usd:.3f} for {len(pilot)} seeds; projected ${projected:.2f} for {len(seeds)} (cap ${CAP:.2f})")
assert projected < CAP - meter.spent_usd, "projected cost is over the cap: stopping before the full run"


## Full run (resumable, 50 seeds per part)

In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError
from isoglot.traindata import generate_parts

def remote(path): return "traindata/" + path.relative_to(OUT).as_posix()
def fetch(path):
    try:
        hf_hub_download(REPO, remote(path), repo_type="dataset", local_dir=OUT.parent)
        print("restored", remote(path))
    except EntryNotFoundError:
        pass
def publish(path):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=remote(path), repo_id=REPO, repo_type="dataset")

raw = generate_parts(client, MODEL, meter, seeds, OUT / "raw", fetch=fetch, publish=publish)
publish(raw)
recs = [json.loads(l) for l in open(raw, encoding="utf-8")]
rows = [r for r in recs if not r.get("skipped")]
print(f"spent ${meter.spent_usd:.2f} | fallbacks {meter.fallbacks} | declined {meter.refusals}")
print(len(rows), "rows |", sum(1 for r in recs if r.get("skipped")), "seeds skipped")
print(collections.Counter((r["kind"], r["lang"]) for r in rows))


## CuratorKIT: clean, deduplicate, export

In [ ]:
from isoglot.traindata import curate

for kind in ("harmful", "benign"):
    result = curate(raw, kind, OUT / "curated")
    result.print_summary()
api.upload_folder(folder_path=str(OUT / "curated"), path_in_repo="traindata/curated",
                  repo_id=REPO, repo_type="dataset")
print("uploaded curated exports ->", REPO, "traindata/curated")
